In [ ]:
# Google Colab setup
!pip install -q polars plotly gdown python-dotenv kailash-kaizen

# Mount Google Drive for datasets
from google.colab import drive
drive.mount("/content/drive")


# ════════════════════════════════════════════════════════════════════════
# ASCENT09 — Exercise 2: Prompt Engineering
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Master zero-shot, few-shot, and chain-of-thought prompting
#   via Delegate, comparing structured output quality across strategies.
#
# TASKS:
#   1. Zero-shot classification with Delegate
#   2. Few-shot with example selection
#   3. Chain-of-thought prompting
#   4. Build a custom Signature for structured extraction
#   5. Compare accuracy across prompting strategies
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import os

import polars as pl

from kaizen_agents import Delegate
from kaizen import Signature, InputField, OutputField
from kaizen_agents.agents.specialized.simple_qa import SimpleQAAgent

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()

if not os.environ.get("OPENAI_API_KEY"):
    print("\u26a0 OPENAI_API_KEY not set \u2014 skipping LLM exercises.")
    print("  Set it in .env to run this exercise with real LLM calls.")
    import sys

    sys.exit(0)

model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
print(f"LLM Model: {model}")



### Data Loading


In [ ]:
loader = ASCENTDataLoader()
reports = loader.load("ascent09", "sg_company_reports.parquet")

sample_docs = reports.head(10)
print(f"Loaded {reports.height:,} documents for classification")

CATEGORIES = ["Financial", "Technology", "Healthcare", "Real Estate", "Manufacturing"]



## TASK 1: Zero-shot classification with Delegate


In [ ]:
async def zero_shot_classify(text: str) -> str:
    """Classify a document using zero-shot prompting.
    Create a Delegate (budget_usd=0.5), build a prompt that lists the CATEGORIES
    and asks the model to respond with ONLY the category name.
    Stream the response and return the stripped text.
    """
    # TODO: Build a zero-shot prompt listing CATEGORIES, create Delegate,
    # stream response, return stripped category string.
    ____
    ____


async def run_zero_shot():
    print(f"\n=== Zero-Shot Classification ===")
    results = []
    texts = sample_docs.select("text").to_series().to_list()
    for i, text in enumerate(texts[:5]):
        category = await zero_shot_classify(text)
        print(f"  Doc {i+1}: {category}")
        results.append(category)
    return results


zero_shot_results  = await run_zero_shot()


## TASK 2: Few-shot with example selection


In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "text": "Revenue increased 15% driven by strong loan growth and net interest margin expansion.",
        "category": "Financial",
    },
    {
        "text": "The company launched its new cloud-native SaaS platform serving enterprise clients across APAC.",
        "category": "Technology",
    },
    {
        "text": "Clinical trials for the new oncology drug showed 40% improvement in patient outcomes.",
        "category": "Healthcare",
    },
    {
        "text": "The integrated township development in Jurong added 2,500 residential units to the portfolio.",
        "category": "Real Estate",
    },
    {
        "text": "Factory automation reduced production cycle time by 30% at the Tuas semiconductor fab.",
        "category": "Manufacturing",
    },
]


async def few_shot_classify(text: str) -> str:
    """Classify using few-shot prompting — prepend FEW_SHOT_EXAMPLES to the prompt.
    Format each example as 'Text: "..." Category: ...' then append the new text.
    Create Delegate (budget_usd=0.5), stream response, return stripped string.
    """
    # TODO: Format FEW_SHOT_EXAMPLES into a prompt prefix, append the new text,
    # create Delegate, stream response.
    ____
    ____


async def run_few_shot():
    print(f"\n=== Few-Shot Classification ===")
    results = []
    texts = sample_docs.select("text").to_series().to_list()
    for i, text in enumerate(texts[:5]):
        category = await few_shot_classify(text)
        print(f"  Doc {i+1}: {category}")
        results.append(category)
    return results


few_shot_results  = await run_few_shot()


## TASK 3: Chain-of-thought prompting


In [ ]:
async def cot_classify(text: str) -> tuple[str, str]:
    """Classify using chain-of-thought: instruct the model to think step by step.
    The prompt should ask: (1) identify key terms, (2) match to category, (3) state final.
    Return (full_reasoning, final_category) where final_category is the last non-empty line.
    """
    # TODO: Build a CoT prompt with explicit reasoning steps, create Delegate
    # (budget_usd=0.5), stream response, extract the last non-empty line as the category.
    ____
    ____


async def run_cot():
    print(f"\n=== Chain-of-Thought Classification ===")
    results = []
    texts = sample_docs.select("text").to_series().to_list()
    for i, text in enumerate(texts[:3]):
        reasoning, category = await cot_classify(text)
        print(f"\n  Doc {i+1} reasoning (excerpt): {reasoning[:200]}...")
        print(f"  Final: {category}")
        results.append(category)
    return results


cot_results  = await run_cot()


## TASK 4: Custom Signature for structured extraction


In [ ]:
# TODO: Define a ReportExtraction Signature with:
#   InputField:  report_text (str)
#   OutputFields: category (str, one of CATEGORIES), key_entities (list[str]),
#                 financial_metrics (list[str]), sentiment (str), confidence (float)
# The Signature is the contract between your code and the LLM — be precise
# about descriptions so the model understands what each field should contain.
____


async def structured_extract():
    """Use SimpleQAAgent with ReportExtraction Signature for structured output."""
    # TODO: Create a SimpleQAAgent with signature=ReportExtraction, model=model,
    # budget_usd=1.0. Run it on the first 3 documents and print the structured fields.
    ____
    ____


structured_results  = await structured_extract()


## TASK 5: Compare accuracy across prompting strategies


In [ ]:
comparison = pl.DataFrame(
    {
        "strategy": ["Zero-Shot", "Few-Shot", "Chain-of-Thought"],
        "doc_1": [
            zero_shot_results[0] if zero_shot_results else "N/A",
            few_shot_results[0] if few_shot_results else "N/A",
            cot_results[0] if cot_results else "N/A",
        ],
        "doc_2": [
            zero_shot_results[1] if len(zero_shot_results) > 1 else "N/A",
            few_shot_results[1] if len(few_shot_results) > 1 else "N/A",
            cot_results[1] if len(cot_results) > 1 else "N/A",
        ],
        "doc_3": [
            zero_shot_results[2] if len(zero_shot_results) > 2 else "N/A",
            few_shot_results[2] if len(few_shot_results) > 2 else "N/A",
            cot_results[2] if len(cot_results) > 2 else "N/A",
        ],
    }
)

print(f"\n=== Strategy Comparison ===")
print(comparison)
print(f"\n  Zero-shot: fast, no examples needed, may hallucinate categories")
print(f"  Few-shot: more consistent, requires curated examples")
print(f"  CoT: best for ambiguous cases, slower and more expensive")
print(f"  Signature: guarantees structure, best for pipelines")

print("\n✓ Exercise 2 complete — prompt engineering strategies compared")

